# Criando o seu pequeno GPT


## Introduction

GPT é um modelo do tipo Decoder, e vamos criar uma versão menor do modelo original (menos blocos de transformers) para ver como gerar texto.

Iremos usar o corpus [simplebooks-92](https://arxiv.org/abs/1911.12391), contendo diversos livros.

> Observação: é recomendado o uso de GPUs para rodar o notebook.

A biblioteca utilizada é o `Keras-NLP`, uma extenção do Keras para modelos mais modernos de NLP.

## Setup

In [ ]:
!pip install -q --upgrade keras-nlp
!pip install -q --upgrade keras  # Upgrade to Keras 3.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.1/792.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.2 MB/s eta 0:00:00


In [ ]:
import os
import keras_nlp
import keras

import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

Hiperparâmetros:

In [ ]:
# Data
BATCH_SIZE = 64
MIN_STRING_LEN = 512  # Strings shorter than this will be discarded
SEQ_LEN = 128  # Length of training sequences, in tokens

# Model
EMBED_DIM = 128
FEED_FORWARD_DIM = 50
NUM_HEADS = 2
NUM_LAYERS = 2
VOCAB_SIZE = 5000  # Limits parameters in model.

# Training
EPOCHS = 3

# Inference
NUM_TOKENS_TO_GENERATE = 20

## Carregando os dados

O corpus `SimpleBooks` tem 1,573 livros do projeto Gutemberg, com vocabulario de aproximadamente ~98k palavras. Não é um corpus grande (como, por exemplo, todas as páginas do Wikipedia), e iremos treinar uma LLM pequena nele (SLM).

In [ ]:
!ls ~/.keras/datasets

simplebooks.zip  simplebooks.zip_archive


In [ ]:
keras.utils.get_file(
    origin="https://dldata-public.s3.us-east-2.amazonaws.com/simplebooks.zip",
    extract=True,
)

In [ ]:
!ls ~/.keras/datasets

simplebooks  simplebooks.zip  simplebooks.zip_archive


Se por alguma razão o aruivo não estiver extraído, rodar abaixo:

In [ ]:
!unzip ~/.keras/datasets/simplebooks.zip_archive -d ~/.keras/datasets/

Archive:  /root/.keras/datasets/simplebooks.zip_archive
   creating: /root/.keras/datasets/simplebooks/
  inflating: /root/.keras/datasets/simplebooks/README.md  
   creating: /root/.keras/datasets/simplebooks/simplebooks-2/
   creating: /root/.keras/datasets/simplebooks/simplebooks-2-raw/
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2-raw/test.txt  
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2-raw/train.txt  
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2-raw/valid.txt  
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2/test.txt  
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2/train.txt  
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2/train.vocab  
  inflating: /root/.keras/datasets/simplebooks/simplebooks-2/valid.txt  
   creating: /root/.keras/datasets/simplebooks/simplebooks-92/
   creating: /root/.keras/datasets/simplebooks/simplebooks-92-raw/
  inflating: /root/.keras/datasets/simplebooks/simpleboo

In [ ]:
dir = os.path.expanduser("~/.keras/datasets/simplebooks/")

# Load simplebooks-92 train set and filter out short lines.
raw_train_ds = (
    tf_data.TextLineDataset(dir + "simplebooks-92-raw/train.txt")
    .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
    .batch(BATCH_SIZE)
    .shuffle(buffer_size=256)
)

# Load simplebooks-92 validation set and filter out short lines.
raw_val_ds = (
    tf_data.TextLineDataset(dir + "simplebooks-92-raw/valid.txt")
    .filter(lambda x: tf_strings.length(x) > MIN_STRING_LEN)
    .batch(BATCH_SIZE)
)

## 1) Treinando o tokenizer


Os modelos GPT usualmente utilizam o algoritmo [Byte-Pair Encoding](https://huggingface.co/learn/nlp-course/en/chapter6/5) para criar um tokenizador que separe os textos em sub-palavras. Porém, nós vamos utilizar o [WordPiece](https://huggingface.co/learn/nlp-course/en/chapter6/6?fw=pt), o mesmo tokenizador utilizado pelo BERT (treinar BPE não está disponível no Keras-NLP).

O algoritmo é recorrente e termina quando chegamos a um tamanho de vocabulário desejado. No nosso caso,  `VOCAB_SIZE` (hiperparâmetro).

A escolha do tamanho do vocabulário é feita considerando que quanto maior o vocabulário, mais custoso e lento é gerar o próximo token, mas um vocabulário pequeno acabará fazendo que muitas palavras sejam consideradas OOV (out of vocabulary).

Além disso, existem algumas palavras especiais que devemos incluir:

- `"[PAD]"` (padding - usamos `SEQ_LEN` de tamanho)
- `"[UNK]"` para OOV sub-words
- `"[BOS]"` (beginning of sentence)

In [ ]:
%%time

# Train tokenizer vocabulary
vocab = keras_nlp.tokenizers.compute_word_piece_vocabulary(
    raw_train_ds,
    vocabulary_size=____,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[BOS]"],
)

CPU times: user 5min 31s, sys: 44.5 s, total: 6min 16s
Wall time: 4min 18s


## Carregando o tokenizador

Usaremos o vocabulário para iniciar uma instância do
`keras_nlp.tokenizers.WordPieceTokenizer`. WordPieceTokenizer irá remover espaços adicionais e mudar o texto para letras minúsculas,  dentre outras transformações.

In [ ]:
tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=SEQ_LEN,
    lowercase=True,
)

## Transformando o texto em tokens

Iremos criar `features` e `labels`.

In [ ]:
# packer adds a start token
start_packer = keras_nlp.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"), # Begin of Sentence
)


def preprocess(inputs):
    outputs = tokenizer(____)
    features = start_packer(outputs)
    labels = outputs
    return features, labels


# Tokenize and split into train and label sequences.
train_ds = raw_train_ds.map(____, num_parallel_calls=tf_data.AUTOTUNE).prefetch(
    tf_data.AUTOTUNE
)
val_ds = raw_val_ds.map(preprocess, num_parallel_calls=tf_data.AUTOTUNE).prefetch(
    tf_data.AUTOTUNE
)

## Criando o modelo com Keras

O modelo tem as seguintes camadas:

- Uma `keras_nlp.layers.TokenAndPositionEmbedding`, wque combina os ebeddings do token e de posição
- Diversas camadas `keras_nlp.layers.TransformerDecoder`, os blocos de decoder do transformers
- Uma camada Dense no final

In [ ]:
inputs = keras.layers.Input(shape=(None,), dtype="int32")

# Embedding.
embedding_layer = keras_nlp.layers.____(
    vocabulary_size=VOCAB_SIZE,
    sequence_length=SEQ_LEN,
    embedding_dim=EMBED_DIM,
    mask_zero=True,
)
x = embedding_layer(inputs)

# Transformer decoders.
for _ in range(____):
    decoder_layer = keras_nlp.layers.TransformerDecoder(
        num_heads=NUM_HEADS,
        intermediate_dim=FEED_FORWARD_DIM,
    )
    x = decoder_layer(x)  # Giving one argument only skips cross-attention.

# Output.
outputs = keras.layers.Dense(VOCAB_SIZE)(x)

model = keras.Model(inputs=inputs, outputs=outputs)

**Sua análise**: O que representa o bloco contido dentro do loop `for` acima? ____

Usaremos a métrica de [Perplexity](https://huggingface.co/docs/transformers/perplexity), muito utilizada para esse tipo de modelo.

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_nlp.metrics.Perplexity(from_logits=True, mask_token_id=0)
model.compile(optimizer="adam", loss=loss_fn, metrics=[perplexity])

In [ ]:
model.summary()

**Sua análise**: Como os parâmetros de distribuem nas camadas? O que isso significa? ____

## Treinando

Basta dar o `.fit()`!

> **Atençao:** Isso pode demorar bastante se não tiver acesso a GPU!!

In [ ]:
%%time

model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

## Usando o modelo

Vamos testar o modelo.

Passando uma semente, por exemplo o token `"[BOS]"`, e geramos os próximos tokens recursivamente.

In [ ]:
# The "packer" layers adds the [BOS] token for us.
prompt_tokens = start_packer(tokenizer([""]))
prompt_tokens

O módulo `keras_nlp.samplers` contém diversos métodos de gerar texto. Eles precisam funções `callback`.

In [ ]:

def next(prompt, cache, index):
    logits = model(____)[:, index - 1, :]
    # Ignore hidden states for now; only needed for contrastive search.
    hidden_states = None
    return logits, hidden_states, cache


Com isso, podemos começar a gerar texto com nosso modelo!

### Greedy search

Esse método utiliza o token com maior probabilidade.

In [ ]:
sampler = keras_nlp.samplers.GreedySampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,  # Start sampling immediately after the [BOS] token.
)
txt = tokenizer.detokenize(output_tokens)
print(f"Greedy search generated text: \n{txt}\n")

**Sua análise**: ____

### Beam search

Esse modelo utiliza `num_beams` para guardar as frases de maior probabilidade (não o token). Isso melhora o texto gerado, mas adciona esfor'co e tempo computacional para gerar cada token.

> **Observação:** beam search com `num_beams=1` é o mesmo que greedy search.

In [ ]:
sampler = keras_nlp.samplers.BeamSampler(num_beams=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Beam search generated text: \n{txt}\n")

**Sua análise**: ____

### Top-K search

Esse método seleciona os `k` tokens com maior probabilidade e escolhe o próximo token aleatoriamente (com probabilidade ajustada) entre eles. Isso evita escolher tokens com probabilidade muito pequena, e por isso os tokens gerados tendem a fazer sentido (se `k` for escolhido adequadamente).

In [ ]:
sampler = keras_nlp.samplers.TopKSampler(k=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-K search generated text: \n{txt}\n")

**Sua análise**: ____

### Top-P search

Com o  método top-k, `k` is fixo, o que torna a escolha de `k` muito relevante, mas pode acabar trazendo tokens indesejados (com probabilidade muito pequena).

Usando uma probabilidade acumulada, podemos escolher entre os tokens que, juntos, obtêm, por exemplo, 90% da probabilidade. Sejam eles 2, ou 10, ou 100!

Dessa forma, o número de tokens que poderão ser escolhidos é dinâmico, evitando os problemas do `top-k`.

In [ ]:
sampler = keras_nlp.samplers.TopPSampler(p=0.5)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-P search generated text: \n{txt}\n")

**Sua análise**: ____